# Databridge: load → verify → (export-ready)

*A runnable tour of the bridge architecture, on synthetic data.*

Machine-learning teams receive object-detection / tracking datasets in many
incompatible on-disk formats and constantly convert between them. **databridge**
turns each input format into one neutral in-memory model, so any input format can
be re-exported to any output format without writing a bespoke converter for every
pair.

This notebook builds a tiny **synthetic** dataset, loads it, validates it, and
shows the loaded model is ready to export — all on generic placeholder data, so it
runs in a clean checkout and is safe to commit (no real dataset content is used).

**The shape of the tool**

```
  loaders             neutral model           converters / writers
  (_formats/hmie/loader.py)     (model.py)              (conversion.py)

  load_hmie  (now)  ─┐                         ┌─► to_mot   (planned)
  load_coco  (plan) ─┼─►   Dataset         ───┼─► to_yolo  (planned)
  load_yolo  (plan) ─┘     VideoSequence       └─► to_coco  (planned)
                           BoxAnnotation
```

Every loader produces the **same** `Dataset`, and every converter consumes
`Dataset` — so any input can feed any output (an N-to-M bridge) rather than a
one-off HMIE→X path. **Today only the HMIE loader (`load_hmie`) and the validator
are implemented; the converters are the next deliverable** (see the roadmap at the
end).

**Terms used here**

- **HMIE / Scale** — the input dataset this loader reads. *Scale* is the
  annotation JSON schema (Scale AI's Video Playback format); *HMIE* is the
  on-disk folder layout it arrives in.
- **AFR** (annotation frame rate) — how many frames per second were labeled,
  which is often lower than the video's native FPS.
- **FMV** — full-motion video (the source clips).
- **MOTChallenge / YOLO / COCO** — common detection/tracking dataset formats; the
  intended export targets.

## Prerequisites

This notebook imports the installed `databridge` package, so install it first
from a clone of the repo, then select that environment as the notebook kernel:

```
poetry install            # enough for this demo (JSON-only, no video decoding)
```

The demo never opens video files, so the optional `video` extra is **not**
required here. You'll want it for the video-probing features mentioned later
(`poetry install --with video`, or `pip install 'databridge[video]'`).

## 0. Build a tiny synthetic HMIE dataset

We write the on-disk HMIE layout that discovery walks: a full-length video
directory, snippet directories, a `scale/` annotation subfolder, and a
`seq_mp4/` video container. Labels, filenames, and ontology URIs are generic
placeholders. Video files are empty placeholders — the default load and the
JSON-only validation never open them (a run with video integrity checks *would*
flag them; see section 2).

This toy tree mirrors the real layout but skips variants real datasets also use
(`seq_ts/` containers, multiple labeler subfolders, batch-level `scale/` dirs).
See the README section **"Dataset layout on disk"** for the full picture.

In [ ]:
import atexit
import json
import shutil
import tempfile
from pathlib import Path


def make_annotation(tracks, *, task_id, afr=5.0, video_fps=30.0):
    """Minimal valid Scale Video Playback annotation dict."""
    annotations = {}
    for i, (label, bbox, n_frames) in enumerate(tracks):
        frames = [
            {
                "keyframeType": "start" if k == 0 else "middle",
                "isInferredKeyframe": False,
                "left": bbox[0],
                "top": bbox[1],
                "width": bbox[2],
                "height": bbox[3],
                "key": k,
                "attributes": {"is_occluded": "0%"},
                "timestamp_secs": k / afr,
            }
            for k in range(n_frames)
        ]
        annotations[f"track-uuid-{i:03d}"] = {
            "label": label,
            "geometry": "box",
            "frames": frames,
        }
    return {
        "task_id": task_id,
        "status": "completed",
        "type": "videoannotation",
        "params": {
            "annotation_frame_rate": afr,
            "videoMetadata": {"video": {"fps": video_fps}},
        },
        "response": {"annotations": annotations, "events": {}},
    }


def make_snippet(full_dir, name, tracks, *, task_id, src="SRC1", h="abc123"):
    """Create one snippet: scale/<CDAO ...>.json + seq_mp4/<name>.mp4."""
    snip = full_dir / name
    (snip / "scale").mkdir(parents=True, exist_ok=True)
    ann = snip / "scale" / f"CDAO_{src}_{name}.mp4_{h}.json"
    ann.write_text(json.dumps(make_annotation(tracks, task_id=task_id), indent=2))
    (snip / "seq_mp4").mkdir(exist_ok=True)
    (snip / "seq_mp4" / f"{name}.mp4").write_bytes(b"")  # placeholder, never opened


# Generic placeholder labels: two plain names + one ontology-style URI. The
# loader treats any string as a label and any http(s) URI as an ontology term.
BOAT = "boat"
VEHICLE = "vehicle"
WIDGET = "http://example.com/ontology/a/FOO_000"

root = Path(tempfile.mkdtemp(prefix="databridge_demo_"))
# Register cleanup now so the temp dir is removed when the kernel exits, even if
# a later cell errors or you stop before the explicit cleanup cell at the end.
atexit.register(shutil.rmtree, root, ignore_errors=True)

full = root / "clip_000000"
full.mkdir(parents=True)
(full / "clip_000000.json").write_text(json.dumps({"data_source": "synthetic"}))

make_snippet(
    full,
    "clip_000001",
    [
        (VEHICLE, (10.0, 20.0, 50.0, 40.0), 5),
        (BOAT, (80.0, 60.0, 30.0, 25.0), 5),
    ],
    task_id="task-0001",
)
make_snippet(
    full,
    "clip_000002",
    [
        (WIDGET, (12.0, 14.0, 44.0, 33.0), 4),
    ],
    task_id="task-0002",
)

# Show the tree we just built.
for p in sorted(root.rglob("*")):
    print(p.relative_to(root))

## 1. Load — `load_hmie(root)` → `Dataset`

One call walks the layout, parses each Scale annotation through the shared
schema, and returns the neutral `Dataset` model. A dataset-wide category map
assigns each ontology label a stable integer id.

In [ ]:
from databridge import load_hmie

ds = load_hmie(root)

print(f"sequences : {len(ds)}")
print(f"boxes     : {ds.num_boxes}")
print(f"categories: {ds.categories}")

### Inspect the neutral model

A `Dataset` holds `VideoSequence`s, each holding `BoxAnnotation`s. These types
live in `databridge.model` and know nothing about HMIE — they are the hub a
converter consumes. A few field semantics worth knowing before you rely on them:

- **`frame_index` is in video-frame space.** The loader maps each label's frame
  `key` (counted at the *annotation* frame rate) onto the video's real frame
  timeline: `frame# = key * fps / afr`. With `afr=5`, `fps=30` the labeled keys
  `0,1,2,…` land on video frames `0,6,12,…` — already aligned for frame-indexed
  formats like MOTChallenge, with no further math.
- **`num_frames` is an estimate unless the video was probed.** By default the
  loader never opens the video, so this is a lower bound (max annotated frame +
  1), not the true clip length. Pass `require_video=True` (needs the `video`
  extra) for the real count.
- **`category_id` is dataset-stable and 1-based**; an unlabeled track gets
  `category_id = -1` and no name. **`category_name`** is the last path segment of
  the label URI (`…/FOO_000` → `FOO_000`), or the label itself when it isn't a URI.

In [ ]:
seq = ds.sequences[0]
print("VideoSequence:")
print(f"  video_id   : {seq.video_id}")
print(f"  fps        : {seq.fps}")
print(f"  num_frames : {seq.num_frames}  (estimate -- not probed)")
print(f"  status     : {seq.status}")
print(f"  boxes      : {len(seq.boxes)}")

box = seq.boxes[0]
print("\nBoxAnnotation (first box):")
print(f"  frame_index   : {box.frame_index}   # mapped to video-frame space")
print(f"  track_id      : {box.track_id}")
print(f"  category_id   : {box.category_id}  ({box.category_name})")
print(f"  bbox (l,t,w,h): {box.bbox}")
print(f"  is_inferred   : {box.is_inferred}")

## 2. Verify — `validate(root)`

Loading is best-effort and deliberately separate from validation. To find out
*why* data is bad, run the validator. Here we skip the video-integrity probe
(`check_video_integrity=False`) since our placeholder mp4s have no real frames;
that is the same JSON-only path the CLI exposes as `--skip-video-check`. Full
video integrity (and `require_video=True`) need the `video` extra.

Note: the label histogram in the summary counts annotated **tracks**, not
per-frame boxes, so it won't match the box total from section 1.

In [ ]:
from databridge import validate

# workers=1 runs the validator serially -- it avoids spawning a process pool,
# which keeps the demo robust inside a notebook kernel. For large real datasets,
# drop it (or raise it) to fan validation across CPU cores.
result = validate(root, check_video_integrity=False, workers=1)
print(result.summary(use_color=False))

## 3. Export readiness (converter is the next step)

The converters are **not implemented yet** — that's the work scheduled below. But
the point of the neutral model is that the export step becomes straightforward:
the fields a frame-indexed, track-centric format like **MOTChallenge** needs are
already present and in the right units. This is the ground-truth `gt.txt` column
layout (the results-file layout differs):

| MOTChallenge `gt.txt` column | source on `BoxAnnotation` |
|------------------------------|---------------------------|
| `frame`                      | `frame_index` (already video-frame space) |
| `id`                         | `track_id` (unique *within a sequence*) |
| `bb_left, bb_top, bb_width, bb_height` | `bbox` (pixels: left, top, w, h) |
| `conf`                       | `1` for ground truth |
| `class`                      | mapped from `category_id` |

A few things the exporter will still own (so "export-ready" stays honest):

- **One file per sequence.** MOT writes a separate `gt.txt` per sequence, so
  `track_id` being unique only *within* a sequence (it resets per sequence) is
  exactly MOT's scope — see the per-sequence output below.
- **`class` is a mapping, not a pass-through.** MOT's `class` is a fixed category
  code; the exporter maps our `category_id` onto it and must decide how to handle
  unlabeled tracks (`category_id = -1`).
- **`is_inferred` / `keyframe_type`** are extra signals the model carries (human
  keyframe vs. tool-interpolated) — optional weighting inputs, not the source of
  `conf`.

The cell below is **not** an exporter — it just prints the fields each box already
exposes, grouped by sequence, to show the model carries what a converter would write.

In [ ]:
# Readiness check only -- NOT the exporter (that lands in conversion.py). Grouped
# by sequence because MOT writes one gt.txt per sequence and track ids are unique
# only within a sequence.
for sequence in ds:
    print(f"# sequence {sequence.video_id}  ({len(sequence.boxes)} boxes)")
    for b in sequence.boxes:
        left, top, wid, hgt = b.bbox
        print(
            f"  frame={b.frame_index:<3} id={b.track_id} "
            f"bbox=({left:.0f},{top:.0f},{wid:.0f},{hgt:.0f}) class={b.category_id}"
        )

In [ ]:
# Optional explicit cleanup. Also registered via atexit in section 0, so the
# temp dataset is removed when the kernel exits even if you skip this cell.
shutil.rmtree(root, ignore_errors=True)
print(f"removed {root}")